In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

REPO_ROOT = Path("..").resolve()
STEP_DIR  = REPO_ROOT / "data/raw/step_responses"
SWEEP_CSV = REPO_ROOT / "data/raw/static_sweep_20260506_run1.csv"

STEP_START_MS = 500
SS_WINDOW_MS  = (4000, 5000)

In [2]:
PATTERN = re.compile(r"step_up_(fwd|rev)_pwm(\d+)_run(\d+)\.csv$")
FIRMWARE_PATTERN = re.compile(r"step_up_(fwd|rev)_(\d+)_run0?(\d+)\.csv$")

trials = {}
for f in sorted(STEP_DIR.glob("step_up_*.csv")):
    m = PATTERN.search(f.name) or FIRMWARE_PATTERN.search(f.name)
    if not m:
        print(f"  SKIP (no match): {f.name}")
        continue
    direction, pwm, run = m.group(1), int(m.group(2)), int(m.group(3))
    trials[(direction, pwm, run)] = pd.read_csv(f)

print(f"Loaded {len(trials)} step-up trials")
print("Conditions:", sorted({(d, p) for (d, p, _) in trials.keys()}))

Loaded 18 step-up trials
Conditions: [('fwd', 160), ('fwd', 200), ('fwd', 240), ('rev', 160), ('rev', 200), ('rev', 240)]


In [3]:
def ss_rpm(df):
    lo, hi = SS_WINDOW_MS
    mask = (df["t_ms"] >= lo + STEP_START_MS) & (df["t_ms"] <= hi + STEP_START_MS)
    return df.loc[mask, "rpm"].mean()

records = []
for (direction, pwm, run), df in trials.items():
    records.append({"direction": direction, "pwm": pwm, "run": run, "rpm_ss": ss_rpm(df)})

ss_df = pd.DataFrame(records)

agg = (ss_df.groupby(["direction", "pwm"])["rpm_ss"]
            .agg(["mean", "std", "count"])
            .reset_index()
            .rename(columns={"mean": "rpm_step_mean", "std": "rpm_step_std"}))
print(agg.to_string(index=False))

direction  pwm  rpm_step_mean  rpm_step_std  count
      fwd  160      52.358959      3.142775      3
      fwd  200     123.885385      1.308000      3
      fwd  240     210.509000      4.706710      3
      rev  160     -69.753217      1.826446      3
      rev  200    -131.836073      0.688417      3
      rev  240    -222.733367      2.513944      3


In [4]:
sweep = pd.read_csv(SWEEP_CSV)
print("Sweep columns:", list(sweep.columns))
print(sweep.head())

Sweep columns: ['pwm_cmd', 'direction', 'vmean_v', 'rpm', 'notes']
   pwm_cmd direction  vmean_v  rpm  \
0       10       fwd   -0.220  0.0   
1       20       fwd   -0.247  0.0   
2       30       fwd   -0.240  0.0   
3       40       fwd   -0.245  0.0   
4       50       fwd   -0.355  0.0   

                                               notes  
0  Ch1=12.2; Ch2=12.4; Math=-220mV; PSU_current_A...  
1  Ch1=11.7; Ch2=11.9; Math=-247mV; PSU_current_A...  
2  Ch1=11.2; Ch2=11.3; Math=-240mV; PSU_current_A...  
3  Ch1=10.7; Ch2=10.8; Math=-245mV; PSU_current_A...  
4  Ch1=3.9-4.1V; Ch2=4.7V; Math=-355mV; PSU_curre...  


In [5]:
sweep_points = sweep[sweep["pwm_cmd"].isin([160, 200, 240])].copy()
sweep_points = sweep_points.rename(columns={"pwm_cmd": "pwm", "rpm": "rpm_sweep"})
sweep_points = sweep_points[["direction", "pwm", "vmean_v", "rpm_sweep"]]

xval = agg.merge(sweep_points, on=["direction", "pwm"], how="inner")
xval["disagreement_pct"] = 100.0 * (xval["rpm_step_mean"].abs() - xval["rpm_sweep"].abs()) / xval["rpm_sweep"].abs()
xval = xval[[
    "direction", "pwm", "vmean_v", "rpm_sweep",
    "rpm_step_mean", "rpm_step_std", "count", "disagreement_pct"
]].sort_values(["pwm", "direction"]).reset_index(drop=True)

print("Cross-val against run01 static sweep:")
print(xval.to_string(index=False))

Cross-val against run01 static sweep:
direction  pwm  vmean_v  rpm_sweep  rpm_step_mean  rpm_step_std  count  disagreement_pct
      fwd  160     2.20       70.0      52.358959      3.142775      3        -25.201488
      rev  160    -3.48      -78.5     -69.753217      1.826446      3        -11.142399
      fwd  200     4.85      130.5     123.885385      1.308000      3         -5.068671
      rev  200    -5.79     -135.0    -131.836073      0.688417      3         -2.343649
      fwd  240     8.41      223.0     210.509000      4.706710      3         -5.601345
      rev  240    -8.93     -225.0    -222.733367      2.513944      3         -1.007393


In [6]:
linear_regime = xval[xval["pwm"] >= 200].copy()
linear_regime["flag_3pct"] = linear_regime["disagreement_pct"].abs() > 3
linear_regime["flag_5pct"] = linear_regime["disagreement_pct"].abs() > 5
linear_regime["flag_7pct"] = linear_regime["disagreement_pct"].abs() > 7

print("Cross-val on above-deadzone region (PWM ≥ 200):")
print(linear_regime.to_string(index=False))

xval.to_csv(REPO_ROOT / "data/processed/phase21_xval_static_consistency.csv", index=False)
print("Saved cross-val table.")

Cross-val on above-deadzone region (PWM ≥ 200):
direction  pwm  vmean_v  rpm_sweep  rpm_step_mean  rpm_step_std  count  disagreement_pct  flag_3pct  flag_5pct  flag_7pct
      fwd  200     4.85      130.5     123.885385      1.308000      3         -5.068671       True       True      False
      rev  200    -5.79     -135.0    -131.836073      0.688417      3         -2.343649      False      False      False
      fwd  240     8.41      223.0     210.509000      4.706710      3         -5.601345       True       True      False
      rev  240    -8.93     -225.0    -222.733367      2.513944      3         -1.007393      False      False      False
Saved cross-val table.
